In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

projec configuration

PROJECT_ROOT = Path("/teamspace/studios/this_studio/spotter-freight-rate")
DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train-test.csv"

RANDOM_SEED = 42

# Model-development months.
# August + September are used to choose/tune the approach.
# October is kept as the final internal holdout after the approach is frozen.
DEVELOPMENT_FOLDS = [
    ("2025-08", "2025-08-01", "2025-09-01"),
    ("2025-09", "2025-09-01", "2025-10-01"),
]

FINAL_INTERNAL_HOLDOUT = ("2025-10", "2025-10-01", "2025-11-01")

# Set this to False later if the history features do not improve the
# development folds. They are target-derived, so we keep them strictly time-safe.
USE_HISTORY_FEATURES = True

print("Configuration loaded.")


Configuration loaded.


## 1. Load and audit the labeled development data

In [ ]:
# load and audit the development enviornment

df = pd.read_csv(TRAIN_PATH)

# Parse date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("=" * 70)
print("DATASET AUDIT")
print("=" * 70)

print(f"Shape: {df.shape}")

if df["date"].notna().any():
    print(
        "Date range:",
        df["date"].min().date(),
        "→",
        df["date"].max().date()
    )

print("Missing dates:", df["date"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate load_ids:", df["load_id"].duplicated().sum())

# required columns

required_columns = [
    "load_id",
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "date",
    "market_index",
    "quote_signal",
    "posted_rate",
]

missing_required = [
    c for c in required_columns
    if c not in df.columns
]

print("\nMissing required columns:", missing_required)

# target audit

print("\n" + "=" * 70)
print("TARGET AUDIT")
print("=" * 70)

print("\nTarget summary:")
display(df["posted_rate"].describe().round(2))

print("\nTarget validity:")
print(
    "Missing posted_rate:",
    df["posted_rate"].isna().sum()
)

print(
    "Non-positive posted_rate:",
    (df["posted_rate"] <= 0).sum()
)

print(
    "Infinite posted_rate:",
    np.isinf(df["posted_rate"]).sum()
)

# numeric feature audit

numeric_cols = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
]

print("\n" + "=" * 70)
print("NUMERIC FEATURE AUDIT")
print("=" * 70)

numeric_audit = pd.DataFrame({
    "dtype": df[numeric_cols].dtypes.astype(str),
    "missing": df[numeric_cols].isna().sum(),
    "negative": [
        (df[c] < 0).sum()
        for c in numeric_cols
    ],
    "infinite": [
        np.isinf(df[c]).sum()
        for c in numeric_cols
    ],
    "unique": [
        df[c].nunique(dropna=True)
        for c in numeric_cols
    ],
})

display(numeric_audit)

# categorical audit

categorical_cols = [
    "pickup",
    "delivery",
    "equipment",
]

print("\n" + "=" * 70)
print("CATEGORICAL FEATURE AUDIT")
print("=" * 70)

for c in categorical_cols:
    print(f"\n--- {c} ---")
    print("Unique:", df[c].nunique(dropna=False))
    print("Missing:", df[c].isna().sum())
    display(
        df[c]
        .value_counts(dropna=False)
        .head(10)
        .rename("count")
        .to_frame()
    )

# data audit

print("\n" + "=" * 70)
print("TEMPORAL AUDIT")
print("=" * 70)

print("Missing dates:", df["date"].isna().sum())

print("\nRows by month:")

monthly_counts = (
    df.dropna(subset=["date"])
      .groupby(df["date"].dt.to_period("M"))
      .size()
      .rename("rows")
      .reset_index()
)

display(monthly_counts)

# basic data ordering check

print("\nRows sorted by date correctly:",
      df["date"].is_monotonic_increasing)

print("\nAudit complete.")

DATASET AUDIT
Shape: (48000, 14)
Date range: 2025-01-01 → 2025-10-31
Missing dates: 0
Duplicate rows: 0
Duplicate load_ids: 0

Missing required columns: []

TARGET AUDIT

Target summary:


count    48000.00
mean      2373.98
std       1486.49
min         57.22
25%       1251.55
50%       2030.76
75%       3330.75
max      25533.00
Name: posted_rate, dtype: float64


Target validity:
Missing posted_rate: 0
Non-positive posted_rate: 0
Infinite posted_rate: 0

NUMERIC FEATURE AUDIT


,dtype,missing,negative,infinite,unique
pickup_lat,float64,0,0,0,64
pickup_lon,float64,0,48000,0,63
delivery_lat,float64,0,0,0,64
delivery_lon,float64,0,48000,0,63
distance,float64,0,0,0,21204
weight,float64,300,292,0,23678
market_index,float64,374,0,0,32884
quote_signal,float64,0,0,0,37633



CATEGORICAL FEATURE AUDIT

--- pickup ---
Unique: 64
Missing: 0


...


--- equipment ---
Unique: 3
Missing: 0


,count
equipment,
Dry Van,27202
Reefer,12045
Flatbed,8753



TEMPORAL AUDIT
Missing dates: 0

Rows by month:


,date,rows
0,2025-01,4918
1,2025-02,4337
2,2025-03,5036
3,2025-04,4819
4,2025-05,4913
5,2025-06,4783
6,2025-07,4912
7,2025-08,4759
8,2025-09,4670
9,2025-10,4853



Rows sorted by date correctly: True

Audit complete.


## 2. Feature engineering

In [ ]:

def haversine_miles(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles."""
    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    return 3958.7613 * 2 * np.arcsin(np.sqrt(a))


def make_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Transform raw load records into leakage-safe, model-ready features.

    Important:
    - We preserve the original weight as weight_raw.
    - Negative weight is NOT silently discarded.
    - weight_clean is NaN for negative values and weight_negative records
      that anomaly explicitly.
    """
    x = frame.copy()

    x["date"] = pd.to_datetime(x["date"], errors="coerce")

    # categorical route

    x["route"] = (
        x["pickup"].astype(str)
        + " -> "
        + x["delivery"].astype(str)
    )

    # weight quality

    x["weight_raw"] = x["weight"]
    x["weight_missing"] = x["weight"].isna().astype("int8")
    x["weight_negative"] = (x["weight"] < 0).astype("int8")
    x["weight_clean"] = x["weight"].where(x["weight"] >= 0)

    # Market quality
    x["market_index_missing"] = x["market_index"].isna().astype("int8")

    # calender features

    x["year"] = x["date"].dt.year
    x["month"] = x["date"].dt.month
    x["day"] = x["date"].dt.day
    x["day_of_week"] = x["date"].dt.dayofweek
    x["day_of_year"] = x["date"].dt.dayofyear
    x["week_of_year"] = x["date"].dt.isocalendar().week.astype("float")

    x["month_sin"] = np.sin(2 * np.pi * x["month"] / 12.0)
    x["month_cos"] = np.cos(2 * np.pi * x["month"] / 12.0)
    x["dow_sin"] = np.sin(2 * np.pi * x["day_of_week"] / 7.0)
    x["dow_cos"] = np.cos(2 * np.pi * x["day_of_week"] / 7.0)
    x["doy_sin"] = np.sin(2 * np.pi * x["day_of_year"] / 365.25)
    x["doy_cos"] = np.cos(2 * np.pi * x["day_of_year"] / 365.25)

    # A monotonic time index can help tree models distinguish trend.
    x["days_since_start"] = (
        x["date"] - pd.Timestamp("2025-01-01")
    ).dt.days

    # non linear numeric representations
    
    x["distance_log1p"] = np.log1p(x["distance"].clip(lower=0))
    x["distance_sq"] = x["distance"] ** 2

    # Preserve original features and add interactions.
    x["distance_x_market"] = x["distance"] * x["market_index"]
    x["distance_x_quote"] = x["distance"] * x["quote_signal"]
    x["market_x_quote"] = x["market_index"] * x["quote_signal"]

    # Geographic relationship between coordinates.
    x["geodesic_distance_miles"] = haversine_miles(
        x["pickup_lat"],
        x["pickup_lon"],
        x["delivery_lat"],
        x["delivery_lon"],
    )

    x["distance_to_geodesic_ratio"] = (
        x["distance"]
        / x["geodesic_distance_miles"].replace(0, np.nan)
    )

    # Fill categoricals explicitly so category handling is stable.
    categorical = ["pickup", "delivery", "equipment", "route"]
    for col in categorical:
        x[col] = x[col].fillna("__MISSING__").astype(str)

    # Date is no longer used directly by the model.
    x = x.drop(
        columns=["load_id", "date", "posted_rate"],
        errors="ignore",
    )

    return x


CATEGORICAL_FEATURES = [
    "pickup",
    "delivery",
    "equipment",
    "route",
]

print("Feature builder ready.")


Feature builder ready.


## 3. Strictly time-safe historical route/origin/destination features

In [ ]:

def add_static_history_features(reference_df, apply_df, high_threshold=8000.0):
    """
    Apply target-derived historical statistics calculated ONLY from reference_df.

    Intended for:
        reference_df = all data strictly BEFORE the validation period
        apply_df     = future validation period

    This avoids using validation targets.
    """
    ref = reference_df.copy()
    out = apply_df.copy()

    ref["route"] = ref["pickup"].astype(str) + " -> " + ref["delivery"].astype(str)
    out["route"] = out["pickup"].astype(str) + " -> " + out["delivery"].astype(str)

    ref["rpm"] = ref["posted_rate"] / ref["distance"].replace(0, np.nan)
    ref["high_rate_flag"] = (ref["posted_rate"] >= high_threshold).astype(float)

    for keys, prefix in [
        (["route"], "route_hist"),
        (["pickup"], "pickup_hist"),
        (["delivery"], "delivery_hist"),
        (["equipment"], "equipment_hist"),
    ]:
        stats = (
            ref.groupby(keys, dropna=False)
            .agg(
                rpm_mean=("rpm", "mean"),
                rate_mean=("posted_rate", "mean"),
                high_rate_share=("high_rate_flag", "mean"),
                history_count=("posted_rate", "count"),
            )
            .reset_index()
            .rename(
                columns={
                    "rpm_mean": f"{prefix}_rpm_mean",
                    "rate_mean": f"{prefix}_rate_mean",
                    "high_rate_share": f"{prefix}_high_rate_share",
                    "history_count": f"{prefix}_count",
                }
            )
        )

        out = out.merge(stats, on=keys, how="left")

    return out


def add_expanding_history_features(training_df, high_threshold=8000.0):
    """
    Build history features for a training fold using only dates PRIOR
    to each row's date.

    Same-day rows do not see one another's targets.
    """
    x = training_df.copy()

    x["route"] = x["pickup"].astype(str) + " -> " + x["delivery"].astype(str)
    x["_rpm"] = x["posted_rate"] / x["distance"].replace(0, np.nan)
    x["_high_rate"] = (x["posted_rate"] >= high_threshold).astype(float)

    base = x[
        ["load_id", "date", "route", "pickup", "delivery", "equipment"]
    ].copy()

    for key_col, prefix in [
        ("route", "route_hist"),
        ("pickup", "pickup_hist"),
        ("delivery", "delivery_hist"),
        ("equipment", "equipment_hist"),
    ]:
        daily = (
            x.groupby([key_col, "date"], dropna=False)
            .agg(
                _rpm_sum=("_rpm", "sum"),
                _rate_sum=("posted_rate", "sum"),
                _high_sum=("_high_rate", "sum"),
                _count=("posted_rate", "count"),
            )
            .reset_index()
            .sort_values([key_col, "date"])
        )

        grouped = daily.groupby(key_col, sort=False)

        daily["_cum_rpm_sum"] = grouped["_rpm_sum"].cumsum()
        daily["_cum_rate_sum"] = grouped["_rate_sum"].cumsum()
        daily["_cum_high_sum"] = grouped["_high_sum"].cumsum()
        daily["_cum_count"] = grouped["_count"].cumsum()

        # Shift after cumulative sum => only dates strictly before current date.
        daily["_prior_rpm_sum"] = grouped["_cum_rpm_sum"].shift(1)
        daily["_prior_rate_sum"] = grouped["_cum_rate_sum"].shift(1)
        daily["_prior_high_sum"] = grouped["_cum_high_sum"].shift(1)
        daily["_prior_count"] = grouped["_cum_count"].shift(1)

        history = daily[
            [key_col, "date",
             "_prior_rpm_sum", "_prior_rate_sum",
             "_prior_high_sum", "_prior_count"]
        ].copy()

        history[f"{prefix}_rpm_mean"] = (
            history["_prior_rpm_sum"]
            / history["_prior_count"].replace(0, np.nan)
        )
        history[f"{prefix}_rate_mean"] = (
            history["_prior_rate_sum"]
            / history["_prior_count"].replace(0, np.nan)
        )
        history[f"{prefix}_high_rate_share"] = (
            history["_prior_high_sum"]
            / history["_prior_count"].replace(0, np.nan)
        )
        history[f"{prefix}_count"] = history["_prior_count"]

        history = history[
            [key_col, "date",
             f"{prefix}_rpm_mean",
             f"{prefix}_rate_mean",
             f"{prefix}_high_rate_share",
             f"{prefix}_count"]
        ]

        base = base.merge(
            history,
            on=[key_col, "date"],
            how="left",
        )

    return base[
        ["load_id"]
        + [
            c for c in base.columns
            if c not in ["load_id", "date", "route", "pickup", "delivery", "equipment"]
        ]
    ].copy()

def attach_history_features(
    raw_train,
    raw_valid,
    use_history=True,
    high_threshold=8000.0,
):
    """
    Build features for one temporal fold.

    Training rows use expanding past-only statistics.
    Validation rows use all labeled history strictly before validation.
    """
    X_train = make_base_features(raw_train)
    X_valid = make_base_features(raw_valid)

    if not use_history:
        return X_train, X_valid

    # Training: prior dates only.
    train_hist = add_expanding_history_features(
        raw_train,
        high_threshold=high_threshold,
    )

    # Validation: aggregate over all historical labeled data.
    valid_hist = add_static_history_features(
        raw_train,
        raw_valid,
        high_threshold=high_threshold,
    )

    train_hist = train_hist.set_index("load_id")
    valid_hist = valid_hist.set_index("load_id")

    # Align history features to the base feature rows by load_id.
    train_hist = train_hist.reindex(raw_train["load_id"]).reset_index(drop=True)
    valid_hist = valid_hist.reindex(raw_valid["load_id"]).reset_index(drop=True)

    history_cols = [
        c for c in train_hist.columns
        if c not in ["load_id"]
    ]

    for c in history_cols:
        X_train[c] = train_hist[c].values
        X_valid[c] = valid_hist[c].values

    return X_train, X_valid


## 4. Metrics — including the metrics needed to diagnose the RMSE problem

In [ ]:
# Regression metrics

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # basic validation

    if len(y_true) != len(y_pred):
        raise ValueError(
            f"Length mismatch: y_true={len(y_true)}, "
            f"y_pred={len(y_pred)}"
        )

    if not np.all(np.isfinite(y_true)):
        raise ValueError("y_true contains NaN or infinite values.")

    if not np.all(np.isfinite(y_pred)):
        raise ValueError("y_pred contains NaN or infinite values.")

    # errors

    abs_error = np.abs(y_true - y_pred)

    pct_error = (
        abs_error
        / np.maximum(np.abs(y_true), 1e-8)
    )

    mean_target = np.mean(y_true)
    median_target = np.median(y_true)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    mape = 100 * np.mean(pct_error)

    return {
        # Primary regression metrics
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape,

        # Error distribution
        "median_abs_error": np.median(abs_error),
        "p95_abs_error": np.percentile(abs_error, 95),

        # Target diagnostics
        "mean_target": mean_target,
        "median_target": median_target,

        # Relative diagnostics
        "RMSE_pct_of_mean": (
            100 * rmse / mean_target
            if mean_target != 0
            else np.nan
        ),

        "MAE_pct_of_mean": (
            100 * mae / mean_target
            if mean_target != 0
            else np.nan
        ),

        # Business-style diagnostic
        "within_15pct": (
            100 * np.mean(pct_error <= 0.15)
        ),
    }


def print_metrics(name, y_true, y_pred):

    m = regression_metrics(y_true, y_pred)

    print(f"\n{name}")
    print("-" * len(name))

    print(f"MAE:                    ${m['MAE']:,.2f}")
    print(f"RMSE:                   ${m['RMSE']:,.2f}")
    print(f"MAPE:                   {m['MAPE']:.2f}%")
    print(f"R²:                     {m['R2']:.4f}")

    print(
        f"Median absolute error:  "
        f"${m['median_abs_error']:,.2f}"
    )

    print(
        f"95th percentile error:  "
        f"${m['p95_abs_error']:,.2f}"
    )

    print(
        f"Mean target:            "
        f"${m['mean_target']:,.2f}"
    )

    print(
        f"Median target:          "
        f"${m['median_target']:,.2f}"
    )

    print(
        f"RMSE / mean target:     "
        f"{m['RMSE_pct_of_mean']:.2f}%"
    )

    print(
        f"MAE / mean target:      "
        f"{m['MAE_pct_of_mean']:.2f}%"
    )

    print(
        f"Predictions within 15%: "
        f"{m['within_15pct']:.2f}%"
    )

    return m



## 5. Model factory

Aligning early-stopping/evaluation with the quantity: **RMSE in the current dollar target**, while still testing an MAE-trained candidate.

Not assuming that MAE improvement means RMSE improvement.
Every candidate is evaluated back on the original dollar target.


In [ ]:
# catboost model factory

def build_catboost(
    loss="RMSE",
    iterations=1200,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=5.0,
    random_strength=0.5,
):
    """
    CatBoost factory.

    Important:
    The outer validation/holdout set is NOT passed here.

    Early stopping is therefore not allowed to inspect
    the outer evaluation period.
    """

    return CatBoostRegressor(
        loss_function=loss,
        eval_metric="RMSE",

        iterations=iterations,
        learning_rate=learning_rate,

        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,

        random_seed=RANDOM_SEED,

        verbose=False,
        allow_writing_files=False,
        thread_count=-1,

        # IMPORTANT:
        # No early_stopping_rounds here.
        #
        # We will determine iterations using an INNER
        # temporal validation period and then freeze them.
    )

In [ ]:
# 5A. SELECT ITERATIONS USING AN INNER TEMPORAL VALIDATION SET

def select_iterations_temporal(
    raw_train,
    inner_valid,
    target_type="direct",
    loss="RMSE",
    use_history=True,
    max_iterations=1600,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=5.0,
    random_strength=0.5,
    high_threshold=8000.0,
):
    """
    Determine the CatBoost iteration count using an INNER
    temporal validation period.

    The outer validation period is never touched here.
    """

    X_train, X_inner = attach_history_features(
        raw_train,
        inner_valid,
        use_history=use_history,
        high_threshold=high_threshold,
    )

    # construct target

    if target_type == "direct":
        y_train = raw_train["posted_rate"].values
        y_inner = inner_valid["posted_rate"].values

    elif target_type == "log":
        y_train = np.log1p(
            raw_train["posted_rate"].values
        )
        y_inner = np.log1p(
            inner_valid["posted_rate"].values
        )

    elif target_type == "rpm":

        train_distance = (
            raw_train["distance"]
            .replace(0, np.nan)
        )

        inner_distance = (
            inner_valid["distance"]
            .replace(0, np.nan)
        )

        y_train = (
            raw_train["posted_rate"]
            / train_distance
        ).values

        y_inner = (
            inner_valid["posted_rate"]
            / inner_distance
        ).values

    else:
        raise ValueError(
            f"Unknown target_type: {target_type}"
        )

    # inner model

    model = CatBoostRegressor(
        loss_function=loss,
        eval_metric="RMSE",

        iterations=max_iterations,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,

        random_seed=RANDOM_SEED,

        verbose=False,
        allow_writing_files=False,
        thread_count=-1,

        # Early stopping is allowed HERE because this is
        # explicitly an INNER validation period.
        early_stopping_rounds=120,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=CATEGORICAL_FEATURES,

        eval_set=(X_inner, y_inner),

        use_best_model=True,
    )

    best_iteration = model.get_best_iteration()

    # CatBoost can return -1 in some edge cases.
    if best_iteration is None or best_iteration < 0:
        best_iteration = max_iterations - 1

    # CatBoost iteration is zero-indexed.
    selected_iterations = max(best_iteration + 1, 50)

    print(
        f"{target_type} / {loss}: "
        f"selected {selected_iterations} iterations"
    )

    return selected_iterations

In [ ]:
# temporal development configurtion

DEVELOPMENT_FOLDS_CLEAN = [
    {
        "name": "2025-08",
        "train_end": "2025-07-01",
        "inner_start": "2025-07-01",
        "inner_end": "2025-08-01",
        "outer_start": "2025-08-01",
        "outer_end": "2025-09-01",
    },
    {
        "name": "2025-09",
        "train_end": "2025-08-01",
        "inner_start": "2025-08-01",
        "inner_end": "2025-09-01",
        "outer_start": "2025-09-01",
        "outer_end": "2025-10-01",
    },
]

## 6. Train one temporal fold and compare target formulations

In [ ]:


def run_clean_temporal_fold(
    raw_train,
    raw_valid,
    selected_iterations,
    use_history=True,
    high_threshold=8000.0,
):
    """
    Train/evaluate candidate target formulations on an OUTER
    temporal validation period.

    CRITICAL:
    raw_valid is used ONLY for final prediction/evaluation.

    It is NOT used for:
        - early stopping
        - iteration selection
        - hyperparameter selection
        - model selection
    """

    X_train, X_valid = attach_history_features(
        raw_train,
        raw_valid,
        use_history=use_history,
        high_threshold=high_threshold,
    )

    models = {}
    predictions = {}
    metrics = []

    y_valid = raw_valid["posted_rate"].values

    # ===========================================================
    # Candidate A: Direct target + RMSE
    # ===========================================================

    model_direct_rmse = build_catboost(
        loss="RMSE",
        iterations=selected_iterations["direct_rmse"],
    )

    model_direct_rmse.fit(
        X_train,
        raw_train["posted_rate"].values,
        cat_features=CATEGORICAL_FEATURES,

        # NO eval_set
        # NO early stopping
    )

    pred_direct_rmse = model_direct_rmse.predict(X_valid)
    pred_direct_rmse = np.maximum(pred_direct_rmse, 1e-6)

    models["direct_rmse"] = model_direct_rmse
    predictions["direct_rmse"] = pred_direct_rmse

    metrics.append({
        "model": "direct_rmse",
        **regression_metrics(
            y_valid,
            pred_direct_rmse,
        ),
    })

    # ===========================================================
    # Candidate B: Direct target + MAE
    # ===========================================================

    model_direct_mae = build_catboost(
        loss="MAE",
        iterations=selected_iterations["direct_mae"],
    )

    model_direct_mae.fit(
        X_train,
        raw_train["posted_rate"].values,
        cat_features=CATEGORICAL_FEATURES,
    )

    pred_direct_mae = model_direct_mae.predict(X_valid)
    pred_direct_mae = np.maximum(
        pred_direct_mae,
        1e-6,
    )

    models["direct_mae"] = model_direct_mae
    predictions["direct_mae"] = pred_direct_mae

    metrics.append({
        "model": "direct_mae",
        **regression_metrics(
            y_valid,
            pred_direct_mae,
        ),
    })

    # ===========================================================
    # Candidate C: Log target + RMSE
    # ===========================================================

    model_log = build_catboost(
        loss="RMSE",
        iterations=selected_iterations["log_rmse"],
    )

    y_train_log = np.log1p(
        raw_train["posted_rate"].values
    )

    model_log.fit(
        X_train,
        y_train_log,
        cat_features=CATEGORICAL_FEATURES,
    )

    pred_log = np.expm1(
        model_log.predict(X_valid)
    )

    pred_log = np.maximum(
        pred_log,
        1e-6,
    )

    models["log_rmse"] = model_log
    predictions["log_rmse"] = pred_log

    metrics.append({
        "model": "log_rmse",
        **regression_metrics(
            y_valid,
            pred_log,
        ),
    })

    # ===========================================================
    # Candidate D: Rate-per-mile + RMSE
    # ===========================================================

    train_distance = (
        raw_train["distance"]
        .replace(0, np.nan)
    )

    train_rpm = (
        raw_train["posted_rate"]
        / train_distance
    ).values

    model_rpm = build_catboost(
        loss="RMSE",
        iterations=selected_iterations["rpm_rmse"],
    )

    model_rpm.fit(
        X_train,
        train_rpm,
        cat_features=CATEGORICAL_FEATURES,
    )

    pred_rpm = (
        model_rpm.predict(X_valid)
        * raw_valid["distance"].values
    )

    pred_rpm = np.maximum(
        pred_rpm,
        1e-6,
    )

    models["rpm_rmse"] = model_rpm
    predictions["rpm_rmse"] = pred_rpm

    metrics.append({
        "model": "rpm_rmse",
        **regression_metrics(
            y_valid,
            pred_rpm,
        ),
    })

    # ===========================================================
    # Results
    # ===========================================================

    metrics_df = (
        pd.DataFrame(metrics)
        .sort_values("RMSE")
        .reset_index(drop=True)
    )

    return {
        "X_train": X_train,
        "X_valid": X_valid,
        "models": models,
        "predictions": predictions,
        "metrics": metrics_df,
        "selected_iterations": selected_iterations,
    }

## 7. Development folds: August and September

In [13]:

# fold_outputs = {}
# all_dev_metrics = []

# for fold_name, start, end in DEVELOPMENT_FOLDS:
#     raw_train = df[df["date"] < pd.Timestamp(start)].copy()
#     raw_valid = df[
#         (df["date"] >= pd.Timestamp(start))
#         & (df["date"] < pd.Timestamp(end))
#     ].copy()

#     print(
#         f"\n===== DEVELOPMENT FOLD {fold_name} ====="
#         f"\nTrain: {len(raw_train):,}"
#         f"\nValid: {len(raw_valid):,}"
#     )

#     result = run_fold(
#         raw_train,
#         raw_valid,
#         use_history=USE_HISTORY_FEATURES,
#     )

#     fold_outputs[fold_name] = result

#     fold_metrics = result["metrics"].copy()
#     fold_metrics["fold"] = fold_name

#     all_dev_metrics.append(fold_metrics)

# dev_metrics = pd.concat(
#     all_dev_metrics,
#     ignore_index=True,
# )

# display(
#     dev_metrics[
#         [
#             "fold",
#             "model",
#             "MAE",
#             "RMSE",
#             "R2",
#             "p95_abs_error",
#             "RMSE_pct_of_mean",
#             "within_15pct",
#         ]
#     ].sort_values(["model", "fold"])
# )
# ===============================================================
# 7. CLEAN DEVELOPMENT FOLDS
# ===============================================================

fold_outputs = {}
all_dev_metrics = []

for fold_cfg in DEVELOPMENT_FOLDS_CLEAN:

    fold_name = fold_cfg["name"]

    print("\n" + "=" * 70)
    print(f"DEVELOPMENT FOLD: {fold_name}")
    print("=" * 70)

    # -----------------------------------------------------------
    # Outer training period
    # -----------------------------------------------------------

    outer_train = df[
        df["date"] < pd.Timestamp(
            fold_cfg["outer_start"]
        )
    ].copy()

    # -----------------------------------------------------------
    # Inner temporal validation
    #
    # This is ONLY used to select iterations.
    # -----------------------------------------------------------

    inner_train = df[
        df["date"] < pd.Timestamp(
            fold_cfg["inner_start"]
        )
    ].copy()

    inner_valid = df[
        (df["date"] >= pd.Timestamp(
            fold_cfg["inner_start"]
        ))
        &
        (df["date"] < pd.Timestamp(
            fold_cfg["inner_end"]
        ))
    ].copy()

    # -----------------------------------------------------------
    # Outer validation
    # -----------------------------------------------------------

    outer_valid = df[
        (df["date"] >= pd.Timestamp(
            fold_cfg["outer_start"]
        ))
        &
        (df["date"] < pd.Timestamp(
            fold_cfg["outer_end"]
        ))
    ].copy()

    print(f"Inner train:   {len(inner_train):,}")
    print(f"Inner valid:   {len(inner_valid):,}")
    print(f"Outer train:   {len(outer_train):,}")
    print(f"Outer valid:   {len(outer_valid):,}")

    # -----------------------------------------------------------
    # Select iterations WITHOUT touching outer validation
    # -----------------------------------------------------------

    selected_iterations = {}

    selected_iterations["direct_rmse"] = (
        select_iterations_temporal(
            inner_train,
            inner_valid,
            target_type="direct",
            loss="RMSE",
            use_history=USE_HISTORY_FEATURES,
        )
    )

    selected_iterations["direct_mae"] = (
        select_iterations_temporal(
            inner_train,
            inner_valid,
            target_type="direct",
            loss="MAE",
            use_history=USE_HISTORY_FEATURES,
        )
    )

    selected_iterations["log_rmse"] = (
        select_iterations_temporal(
            inner_train,
            inner_valid,
            target_type="log",
            loss="RMSE",
            use_history=USE_HISTORY_FEATURES,
        )
    )

    selected_iterations["rpm_rmse"] = (
        select_iterations_temporal(
            inner_train,
            inner_valid,
            target_type="rpm",
            loss="RMSE",
            use_history=USE_HISTORY_FEATURES,
        )
    )

    print("\nSelected iterations:")
    print(selected_iterations)

    # -----------------------------------------------------------
    # Evaluate frozen iteration counts on OUTER period
    # -----------------------------------------------------------

    result = run_clean_temporal_fold(
        outer_train,
        outer_valid,
        selected_iterations=selected_iterations,
        use_history=USE_HISTORY_FEATURES,
    )

    fold_outputs[fold_name] = result

    fold_metrics = result["metrics"].copy()
    fold_metrics["fold"] = fold_name

    all_dev_metrics.append(fold_metrics)

    display(
        fold_metrics[
            [
                "fold",
                "model",
                "MAE",
                "RMSE",
                "MAPE",
                "R2",
                "p95_abs_error",
                "RMSE_pct_of_mean",
                "within_15pct",
            ]
        ]
    )

# ---------------------------------------------------------------
# Combine development metrics
# ---------------------------------------------------------------

dev_metrics = pd.concat(
    all_dev_metrics,
    ignore_index=True,
)

print("\n" + "=" * 70)
print("CLEAN DEVELOPMENT RESULTS")
print("=" * 70)

display(
    dev_metrics[
        [
            "fold",
            "model",
            "MAE",
            "RMSE",
            "MAPE",
            "R2",
            "p95_abs_error",
            "RMSE_pct_of_mean",
            "within_15pct",
        ]
    ]
    .sort_values(["model", "fold"])
)


DEVELOPMENT FOLD: 2025-08
Inner train:   28,806
Inner valid:   4,912
Outer train:   33,718
Outer valid:   4,759


direct / RMSE: selected 95 iterations
direct / MAE: selected 150 iterations
log / RMSE: selected 160 iterations
rpm / RMSE: selected 111 iterations

Selected iterations:
{'direct_rmse': 95, 'direct_mae': 150, 'log_rmse': 160, 'rpm_rmse': 111}


,fold,model,MAE,RMSE,MAPE,R2,p95_abs_error,RMSE_pct_of_mean,within_15pct
0,2025-08,log_rmse,107.183314,615.895748,5.078887,0.825605,187.958595,26.338254,98.361000
1,2025-08,direct_rmse,139.411970,619.520630,8.598774,0.823546,251.414575,26.493269,93.675142
2,2025-08,direct_mae,124.191954,619.880957,7.517798,0.823341,240.679368,26.508678,93.254885
3,2025-08,rpm_rmse,171.949757,628.122302,7.365131,0.818612,396.511715,26.861112,98.087834



DEVELOPMENT FOLD: 2025-09
Inner train:   33,718
Inner valid:   4,759
Outer train:   38,477
Outer valid:   4,670
direct / RMSE: selected 94 iterations
direct / MAE: selected 238 iterations
log / RMSE: selected 457 iterations
rpm / RMSE: selected 244 iterations

Selected iterations:
{'direct_rmse': 94, 'direct_mae': 238, 'log_rmse': 457, 'rpm_rmse': 244}


,fold,model,MAE,RMSE,MAPE,R2,p95_abs_error,RMSE_pct_of_mean,within_15pct
0,2025-09,rpm_rmse,98.191460,614.520541,4.065780,0.837262,172.300644,25.537200,98.394004
1,2025-09,log_rmse,121.630383,621.685789,5.002034,0.833444,250.997013,25.834961,98.458244
2,2025-09,direct_rmse,129.289528,621.960741,6.912479,0.833297,273.780771,25.846387,95.160600
3,2025-09,direct_mae,140.570911,630.134542,5.646176,0.828887,313.416510,26.186060,97.987152



CLEAN DEVELOPMENT RESULTS


,fold,model,MAE,RMSE,MAPE,R2,p95_abs_error,RMSE_pct_of_mean,within_15pct
2,2025-08,direct_mae,124.191954,619.880957,7.517798,0.823341,240.679368,26.508678,93.254885
7,2025-09,direct_mae,140.570911,630.134542,5.646176,0.828887,313.416510,26.186060,97.987152
1,2025-08,direct_rmse,139.411970,619.520630,8.598774,0.823546,251.414575,26.493269,93.675142
6,2025-09,direct_rmse,129.289528,621.960741,6.912479,0.833297,273.780771,25.846387,95.160600
0,2025-08,log_rmse,107.183314,615.895748,5.078887,0.825605,187.958595,26.338254,98.361000
5,2025-09,log_rmse,121.630383,621.685789,5.002034,0.833444,250.997013,25.834961,98.458244
3,2025-08,rpm_rmse,171.949757,628.122302,7.365131,0.818612,396.511715,26.861112,98.087834
4,2025-09,rpm_rmse,98.191460,614.520541,4.065780,0.837262,172.300644,25.537200,98.394004



## 8. Primary model-development decision

Do **not** select a model from one month.

For the development phase:

1. Compare the mean RMSE across August + September.
2. Use MAE as a guardrail so RMSE improvements do not come from simply making normal predictions worse.
3. Inspect the 95th-percentile error because the current problem is dominated by a small tail.
4. Use the October month only after this approach is locked.

There is no official “15% error” requirement in the supplied assessment instructions. We track `RMSE / mean target` and `% within 15%` as diagnostics, not as official Spotter metrics.


In [14]:

# development_summary = (
#     dev_metrics
#     .groupby("model", as_index=False)
#     .agg(
#         mean_MAE=("MAE", "mean"),
#         mean_RMSE=("RMSE", "mean"),
#         mean_R2=("R2", "mean"),
#         mean_p95_error=("p95_abs_error", "mean"),
#         mean_RMSE_pct=("RMSE_pct_of_mean", "mean"),
#         mean_within_15pct=("within_15pct", "mean"),
#     )
#     .sort_values("mean_RMSE")
# )

# display(development_summary)
# ===============================================================
# 8. DEVELOPMENT MODEL COMPARISON
# ===============================================================

development_summary = (
    dev_metrics
    .groupby("model", as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        mean_MAPE=("MAPE", "mean"),
        mean_R2=("R2", "mean"),
        mean_median_abs_error=(
            "median_abs_error",
            "mean",
        ),
        mean_p95_error=(
            "p95_abs_error",
            "mean",
        ),
        mean_RMSE_pct=(
            "RMSE_pct_of_mean",
            "mean",
        ),
        mean_within_15pct=(
            "within_15pct",
            "mean",
        ),
    )
    .sort_values("mean_RMSE")
    .reset_index(drop=True)
)

display(development_summary)

,model,mean_MAE,mean_RMSE,mean_MAPE,mean_R2,mean_median_abs_error,mean_p95_error,mean_RMSE_pct,mean_within_15pct
0,log_rmse,114.406848,618.790769,5.040460,0.829525,44.589925,219.477804,26.086607,98.409622
1,direct_rmse,134.350749,620.740685,7.755626,0.828422,67.489949,262.597673,26.169828,94.417871
2,rpm_rmse,135.070608,621.321421,5.715456,0.827937,59.451664,284.406179,26.199156,98.240919
3,direct_mae,132.381433,625.007749,6.581987,0.826114,52.284424,277.047939,26.347369,95.621019



## 9. Blend only models that are actually complementary

We do not hard-code a “60/40 CatBoost/XGBoost” blend.

Instead, if two target formulations have different errors, search blend weights **only on the August + September development predictions**.

October remains untouched for this decision.


In [15]:

# # Collect aligned development predictions for ensemble-weight search.
# blend_rows = []

# fold_lookup = {
#     name: (start, end)
#     for name, start, end in DEVELOPMENT_FOLDS
# }

# for fold_name, result in fold_outputs.items():
#     start, end = fold_lookup[fold_name]

#     raw_train = df[
#         df["date"] < pd.Timestamp(start)
#     ].copy()

#     raw_valid = df[
#         (df["date"] >= pd.Timestamp(start))
#         & (df["date"] < pd.Timestamp(end))
#     ].copy()

#     blend_rows.append({
#         "actual": raw_valid["posted_rate"].values,
#         "direct_mae": result["predictions"]["direct_mae"],
#         "log_rmse": result["predictions"]["log_rmse"],
#         "direct_rmse": result["predictions"]["direct_rmse"],
#         "rpm_rmse": result["predictions"]["rpm_rmse"],
#     })

# def concat_pred(key):
#     return np.concatenate([x[key] for x in blend_rows])

# y_blend = np.concatenate([x["actual"] for x in blend_rows])

# blend_candidates = []

# # Two-model blends first.
# for w in np.linspace(0, 1, 21):
#     pred = (
#         w * concat_pred("direct_mae")
#         + (1 - w) * concat_pred("log_rmse")
#     )

#     m = regression_metrics(y_blend, pred)

#     blend_candidates.append({
#         "blend": "direct_mae + log_rmse",
#         "weight_direct_mae": w,
#         "weight_log_rmse": 1 - w,
#         "RMSE": m["RMSE"],
#         "MAE": m["MAE"],
#         "R2": m["R2"],
#         "p95_abs_error": m["p95_abs_error"],
#     })

# blend_table = (
#     pd.DataFrame(blend_candidates)
#     .sort_values(["RMSE", "MAE"])
# )

# display(blend_table.head(10))
blend_rows = []

for fold_cfg in DEVELOPMENT_FOLDS_CLEAN:

    fold_name = fold_cfg["name"]

    result = fold_outputs[fold_name]

    outer_valid = df[
        (df["date"] >= pd.Timestamp(
            fold_cfg["outer_start"]
        ))
        &
        (df["date"] < pd.Timestamp(
            fold_cfg["outer_end"]
        ))
    ].copy()

    blend_rows.append({
        "actual": outer_valid["posted_rate"].values,

        "direct_mae":
            result["predictions"]["direct_mae"],

        "log_rmse":
            result["predictions"]["log_rmse"],

        "direct_rmse":
            result["predictions"]["direct_rmse"],

        "rpm_rmse":
            result["predictions"]["rpm_rmse"],
    })


def concat_pred(key):
    return np.concatenate(
        [row[key] for row in blend_rows]
    )


y_blend = np.concatenate(
    [row["actual"] for row in blend_rows]
)


# ===============================================================
# Two-model blend search
# ===============================================================

blend_candidates = []

for w in np.linspace(0, 1, 21):

    pred = (
        w * concat_pred("direct_mae")
        + (1 - w) * concat_pred("log_rmse")
    )

    m = regression_metrics(
        y_blend,
        pred,
    )

    blend_candidates.append({
        "blend": "direct_mae + log_rmse",

        "weight_direct_mae": w,
        "weight_log_rmse": 1 - w,

        "RMSE": m["RMSE"],
        "MAE": m["MAE"],
        "MAPE": m["MAPE"],
        "R2": m["R2"],
        "p95_abs_error": m["p95_abs_error"],
    })


blend_table = (
    pd.DataFrame(blend_candidates)
    .sort_values(
        ["RMSE", "MAE"]
    )
    .reset_index(drop=True)
)

display(
    blend_table.head(10)
)

,blend,weight_direct_mae,weight_log_rmse,RMSE,MAE,MAPE,R2,p95_abs_error
0,direct_mae + log_rmse,0.00,1.00,618.770214,114.338666,5.040823,0.829702,216.133578
1,direct_mae + log_rmse,0.05,0.95,618.851324,114.134372,5.033025,0.829657,216.549838
2,direct_mae + log_rmse,0.10,0.90,618.956727,114.109329,5.043853,0.829599,217.290381
3,direct_mae + log_rmse,0.15,0.85,619.086411,114.226726,5.068501,0.829528,218.804055
4,direct_mae + log_rmse,0.20,0.80,619.240361,114.478963,5.104713,0.829443,222.099489
5,direct_mae + log_rmse,0.25,0.75,619.418558,114.838785,5.149499,0.829345,222.864905
6,direct_mae + log_rmse,0.30,0.70,619.620981,115.306930,5.201270,0.829234,223.246271
7,direct_mae + log_rmse,0.35,0.65,619.847607,115.886385,5.260578,0.829109,226.099884
8,direct_mae + log_rmse,0.40,0.60,620.098410,116.572287,5.326824,0.828970,228.584433
9,direct_mae + log_rmse,0.45,0.55,620.373359,117.371509,5.400109,0.828819,231.751258


In [16]:
# ===============================================================
# BLEND VS SINGLE MODELS
# ===============================================================

single_model_results = []

for model_name in [
    "direct_mae",
    "log_rmse",
    "direct_rmse",
    "rpm_rmse",
]:

    pred = concat_pred(model_name)

    m = regression_metrics(
        y_blend,
        pred,
    )

    single_model_results.append({
        "model": model_name,
        "RMSE": m["RMSE"],
        "MAE": m["MAE"],
        "MAPE": m["MAPE"],
        "R2": m["R2"],
        "p95_abs_error": m["p95_abs_error"],
    })


single_table = pd.DataFrame(
    single_model_results
)

display(single_table.sort_values("RMSE"))

print("\nBest blend:")
display(blend_table.head(1))

,model,RMSE,MAE,MAPE,R2,p95_abs_error
1,log_rmse,618.770214,114.338666,5.040823,0.829702,216.133578
2,direct_rmse,620.730368,134.398522,7.763585,0.828621,260.231503
3,rpm_rmse,621.422827,135.418709,5.731027,0.828239,327.708537
0,direct_mae,624.980384,132.304132,6.590821,0.826267,280.916051



Best blend:


,blend,weight_direct_mae,weight_log_rmse,RMSE,MAE,MAPE,R2,p95_abs_error
0,direct_mae + log_rmse,0.0,1.0,618.770214,114.338666,5.040823,0.829702,216.133578


In [17]:
# ===============================================================
# 10. FREEZE MODEL DEVELOPMENT DECISION
# ===============================================================

# Clean August + September OOS development results showed:
# log_rmse performed best among the individual models.
# The best tested blend assigned 100% weight to log_rmse,
# so there is no additional benefit from blending.

SELECTED_TARGET = "log_rmse"

SELECTED_BLEND = {
    "direct_mae": 0.0,
    "log_rmse": 1.0,
}

print("=" * 70)
print("FROZEN MODEL DEVELOPMENT DECISION")
print("=" * 70)

print("Selected target/model:", SELECTED_TARGET)
print("Selected blend:", SELECTED_BLEND)
print("Blend effectively reduces to: log_rmse")
print("\nModel development is now FROZEN.")
print("October will be used only as the final internal holdout.")

FROZEN MODEL DEVELOPMENT DECISION
Selected target/model: log_rmse
Selected blend: {'direct_mae': 0.0, 'log_rmse': 1.0}
Blend effectively reduces to: log_rmse

Model development is now FROZEN.
October will be used only as the final internal holdout.


In [18]:
# ===============================================================
# 11. FINAL INTERNAL HOLDOUT — OCTOBER
# ===============================================================

oct_train = df[
    df["date"] < pd.Timestamp("2025-10-01")
].copy()

oct_valid = df[
    (df["date"] >= pd.Timestamp("2025-10-01"))
    &
    (df["date"] < pd.Timestamp("2025-11-01"))
].copy()

print("=" * 70)
print("FINAL INTERNAL HOLDOUT — OCTOBER")
print("=" * 70)

print("October train rows:", len(oct_train))
print("October holdout rows:", len(oct_valid))

FINAL INTERNAL HOLDOUT — OCTOBER
October train rows: 43147
October holdout rows: 4853


In [19]:
oct_inner_train = df[
    df["date"] < pd.Timestamp("2025-09-01")
].copy()

oct_inner_valid = df[
    (df["date"] >= pd.Timestamp("2025-09-01"))
    &
    (df["date"] < pd.Timestamp("2025-10-01"))
].copy()

In [20]:
oct_selected_iterations = {}

oct_selected_iterations["direct_rmse"] = (
    select_iterations_temporal(
        oct_inner_train,
        oct_inner_valid,
        target_type="direct",
        loss="RMSE",
        use_history=USE_HISTORY_FEATURES,
    )
)

oct_selected_iterations["direct_mae"] = (
    select_iterations_temporal(
        oct_inner_train,
        oct_inner_valid,
        target_type="direct",
        loss="MAE",
        use_history=USE_HISTORY_FEATURES,
    )
)

oct_selected_iterations["log_rmse"] = (
    select_iterations_temporal(
        oct_inner_train,
        oct_inner_valid,
        target_type="log",
        loss="RMSE",
        use_history=USE_HISTORY_FEATURES,
    )
)

oct_selected_iterations["rpm_rmse"] = (
    select_iterations_temporal(
        oct_inner_train,
        oct_inner_valid,
        target_type="rpm",
        loss="RMSE",
        use_history=USE_HISTORY_FEATURES,
    )
)

# for only log rmse
# print("\nFrozen October iteration counts:")
# print(oct_selected_iterations)

# oct_selected_iterations = {}

# oct_selected_iterations["log_rmse"] = (
#     select_iterations_temporal(
#         oct_inner_train,
#         oct_inner_valid,
#         target_type="log",
#         loss="RMSE",
#         use_history=USE_HISTORY_FEATURES,
#     )
# )

# print("\nFrozen October iteration count:")
# print(oct_selected_iterations)

direct / RMSE: selected 221 iterations
direct / MAE: selected 493 iterations
log / RMSE: selected 178 iterations
rpm / RMSE: selected 79 iterations


In [21]:
# ===============================================================
# 12. FINAL INTERNAL HOLDOUT — TRAIN FROZEN LOG-RMSE MODEL
# ===============================================================

print("=" * 70)
print("TRAINING FROZEN LOG-RMSE MODEL — JAN-SEP")
print("=" * 70)

# Build history features using ONLY Jan-Sep labeled data.
X_oct_train, X_oct_valid = attach_history_features(
    oct_train,
    oct_valid,
    use_history=USE_HISTORY_FEATURES,
    high_threshold=8000.0,
)

# Log-transformed target
y_oct_train = np.log1p(
    oct_train["posted_rate"].values
)

# Selected ONLY from the inner temporal validation:
# Jan-Aug -> September
final_oct_iterations = oct_selected_iterations["log_rmse"]

print("Selected iterations:", final_oct_iterations)
print("Training rows:", len(oct_train))
print("October holdout rows:", len(oct_valid))

# IMPORTANT:
# No October eval_set.
# No early stopping.
oct_model = build_catboost(
    loss="RMSE",
    iterations=final_oct_iterations,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=5.0,
    random_strength=0.5,
)

oct_model.fit(
    X_oct_train,
    y_oct_train,
    cat_features=CATEGORICAL_FEATURES,
)

print("\nModel training complete.")

TRAINING FROZEN LOG-RMSE MODEL — JAN-SEP


Selected iterations: 178
Training rows: 43147
October holdout rows: 4853

Model training complete.


In [22]:
# ===============================================================
# 13. OCTOBER FINAL INTERNAL EVALUATION
# ===============================================================

log_pred_oct = oct_model.predict(X_oct_valid)

# Convert log predictions back to dollar-rate scale
pred_oct = np.expm1(log_pred_oct)

# Safety: rates must be positive
pred_oct = np.maximum(pred_oct, 1e-6)

y_oct = oct_valid["posted_rate"].values

oct_metrics = regression_metrics(
    y_oct,
    pred_oct,
)

print("=" * 70)
print("FINAL INTERNAL HOLDOUT — OCTOBER RESULTS")
print("=" * 70)

for metric, value in oct_metrics.items():
    print(f"{metric:20s}: {value:.6f}")

FINAL INTERNAL HOLDOUT — OCTOBER RESULTS
MAE                 : 121.040041
RMSE                : 650.379012
R2                  : 0.818962
MAPE                : 5.629690
median_abs_error    : 43.289144
p95_abs_error       : 200.960541
mean_target         : 2379.051374
median_target       : 2035.900000
RMSE_pct_of_mean    : 27.337746
MAE_pct_of_mean     : 5.087744
within_15pct        : 98.104265


In [23]:
# ===============================================================
# 14. SAVE OCTOBER HOLDOUT PREDICTIONS
# ===============================================================

oct_results = oct_valid[
    [
        "load_id",
        "date",
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "posted_rate",
    ]
].copy()

oct_results["predicted_rate"] = pred_oct
oct_results["absolute_error"] = np.abs(
    oct_results["posted_rate"] -
    oct_results["predicted_rate"]
)

oct_results["percentage_error"] = (
    oct_results["absolute_error"] /
    np.maximum(oct_results["posted_rate"], 1e-6)
) * 100

print("=" * 70)
print("OCTOBER ERROR SUMMARY")
print("=" * 70)

display(
    oct_results[
        [
            "posted_rate",
            "predicted_rate",
            "absolute_error",
            "percentage_error",
        ]
    ].describe()
)

OCTOBER ERROR SUMMARY


,posted_rate,predicted_rate,absolute_error,percentage_error
count,4853.000000,4853.000000,4853.000000,4853.000000
mean,2379.051374,2335.815221,121.040041,5.629690
std,1528.714038,1364.981389,639.082411,28.597479
min,104.380000,229.806567,0.005559,0.000266
25%,1231.130000,1224.396889,17.663405,1.086886
50%,2035.900000,1998.985448,43.289144,2.254723
75%,3305.450000,3309.772394,86.915314,3.817231
max,19110.300000,6400.728694,13810.655396,476.172452


## 10. Final internal holdout: October

In [24]:
# ===============================================================
# 15. OCTOBER ERROR BY TARGET RANGE
# ===============================================================

oct_results["target_bucket"] = pd.cut(
    oct_results["posted_rate"],
    bins=[0, 2000, 5000, 8000, 12000, np.inf],
    labels=[
        "<=2k",
        "2k-5k",
        "5k-8k",
        "8k-12k",
        ">12k",
    ],
)

oct_bucket_metrics = (
    oct_results
    .groupby("target_bucket", observed=False)
    .apply(
        lambda g: pd.Series({
            "rows": len(g),
            "MAE": np.mean(
                np.abs(g["posted_rate"] - g["predicted_rate"])
            ),
            "RMSE": np.sqrt(
                np.mean(
                    (g["posted_rate"] - g["predicted_rate"]) ** 2
                )
            ),
        })
    )
    .reset_index()
)

display(oct_bucket_metrics)

,target_bucket,rows,MAE,RMSE
0,<=2k,2376.0,59.154086,246.664949
1,2k-5k,2219.0,95.594195,220.942468
2,5k-8k,238.0,254.200735,759.843812
3,8k-12k,12.0,6914.764694,7057.461285
4,>12k,8.0,11407.092524,11536.858217


In [25]:
tail_check = (
    oct_results
    .groupby("target_bucket", observed=False)
    .apply(
        lambda g: pd.Series({
            "rows": len(g),
            "mean_actual": g["posted_rate"].mean(),
            "mean_prediction": g["predicted_rate"].mean(),
            "MAE": np.mean(
                np.abs(
                    g["posted_rate"] -
                    g["predicted_rate"]
                )
            ),
            "RMSE": np.sqrt(
                np.mean(
                    (
                        g["posted_rate"] -
                        g["predicted_rate"]
                    ) ** 2
                )
            ),
            "median_abs_error": np.median(
                np.abs(
                    g["posted_rate"] -
                    g["predicted_rate"]
                )
            ),
        })
    )
    .reset_index()
)

display(tail_check)

,target_bucket,rows,mean_actual,mean_prediction,MAE,RMSE,median_abs_error
0,<=2k,2376.0,1215.545152,1228.128139,59.154086,246.664949,24.796900
1,2k-5k,2219.0,3199.697413,3193.877479,95.594195,220.942468,69.628080
2,5k-8k,238.0,5520.278992,5299.380343,254.200735,759.843812,108.016555
3,8k-12k,12.0,9701.455000,2786.690306,6914.764694,7057.461285,6208.159298
4,>12k,8.0,15878.577500,4471.484976,11407.092524,11536.858217,10594.989459


In [26]:
worst_october = (
    oct_results
    .sort_values("absolute_error", ascending=False)
    [
        [
            "load_id",
            "date",
            "pickup",
            "delivery",
            "distance",
            "equipment",
            "weight",
            "posted_rate",
            "predicted_rate",
            "absolute_error",
            "percentage_error",
        ]
    ]
    .head(10)
)

display(worst_october)

,load_id,date,pickup,delivery,distance,equipment,weight,posted_rate,predicted_rate,absolute_error,percentage_error
43539,TR-043540,2025-10-03,Milwaukee,Bakersfield,1893.0,Dry Van,30760.0,17514.11,3703.454604,13810.655396,78.854452
46213,TR-046214,2025-10-20,Columbia,Tucson,2023.4,Flatbed,34710.0,17893.17,4283.720740,13609.449260,76.059464
47298,TR-047299,2025-10-27,Boston,Bakersfield,2979.1,Reefer,30585.0,19110.30,5874.560336,13235.739664,69.259717
46516,TR-046517,2025-10-22,Toledo,Albuquerque,1684.3,Reefer,31536.0,14784.38,3620.858790,11163.521210,75.508890
47536,TR-047537,2025-10-29,Los Angeles,Richmond,2782.4,Dry Van,25730.0,15003.55,4977.092292,10026.457708,66.827236
46366,TR-046367,2025-10-21,Bakersfield,Columbia,2251.9,Dry Van,40729.0,14403.14,4389.315758,10013.824242,69.525286
44267,TR-044268,2025-10-08,Albany,Albuquerque,2492.6,Dry Van,25513.0,14425.42,4500.681942,9924.738058,68.800340
43302,TR-043303,2025-10-01,Atlanta,Phoenix,2083.1,Reefer,40335.0,13894.55,4422.195347,9472.354653,68.173166
43384,TR-043385,2025-10-02,Lexington,Lubbock,1187.0,Dry Van,25359.0,11696.44,2353.003057,9343.436943,79.882742
47586,TR-047587,2025-10-29,Corpus Christi,Las Vegas,1383.3,Flatbed,18645.0,11596.28,2865.358226,8730.921774,75.290712


In [27]:
# ===============================================================
# 16. FINAL MODEL CONFIGURATION — FROZEN
# ===============================================================

FINAL_MODEL_CONFIG = {
    "model": "CatBoostRegressor",
    "target": "log1p(posted_rate)",
    "loss": "RMSE",
    "iterations": int(final_oct_iterations),
    "learning_rate": 0.04,
    "depth": 8,
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "history_features": USE_HISTORY_FEATURES,
    "development_validation": "expanding temporal validation",
    "final_internal_holdout": "October 2025",
}

print("=" * 70)
print("FINAL MODEL CONFIGURATION")
print("=" * 70)

for key, value in FINAL_MODEL_CONFIG.items():
    print(f"{key:25s}: {value}")

FINAL MODEL CONFIGURATION
model                    : CatBoostRegressor
target                   : log1p(posted_rate)
loss                     : RMSE
iterations               : 178
learning_rate            : 0.04
depth                    : 8
l2_leaf_reg              : 5.0
random_strength          : 0.5
history_features         : True
development_validation   : expanding temporal validation
final_internal_holdout   : October 2025


In [28]:
final_train = df[
    df["date"] < pd.Timestamp("2025-11-01")
].copy()

print("=" * 70)
print("FINAL PRODUCTION TRAINING DATA")
print("=" * 70)

print("Training rows:", len(final_train))
print("Date range:", final_train["date"].min(), "to", final_train["date"].max())
print("Target missing:", final_train["posted_rate"].isna().sum())

FINAL PRODUCTION TRAINING DATA
Training rows: 48000
Date range: 2025-01-01 00:00:00 to 2025-10-31 00:00:00
Target missing: 0


In [29]:
validation_df = pd.read_csv(
    "data/validation.csv"
)

validation_template = pd.read_csv(
    "data/validation-predictions-template.csv"
)

# Ensure dates are datetime
validation_df["date"] = pd.to_datetime(
    validation_df["date"],
    errors="coerce"
)

print("=" * 70)
print("FINAL VALIDATION DATA")
print("=" * 70)

print("validation.csv rows:", len(validation_df))
print("template rows:", len(validation_template))

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTemplate columns:")
print(validation_template.columns.tolist())

FINAL VALIDATION DATA
validation.csv rows: 12000
template rows: 12000

Validation columns:
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

Template columns:
['load_id', 'predicted_rate']


In [30]:
assert len(validation_df) == 12000
assert len(validation_template) == 12000

assert validation_df["load_id"].is_unique
assert validation_template["load_id"].is_unique

assert validation_df["load_id"].tolist() == \
       validation_template["load_id"].tolist()

assert list(validation_template.columns) == [
    "load_id",
    "predicted_rate"
]

print("Template and validation load IDs match exactly.")
print("Template structure verified.")

Template and validation load IDs match exactly.
Template structure verified.


In [31]:
# ===============================================================
# 18. FINAL PRODUCTION FEATURES
# ===============================================================

X_final_train, X_final_valid = attach_history_features(
    final_train,
    validation_df,
    use_history=USE_HISTORY_FEATURES,
    high_threshold=8000.0,
)

print("=" * 70)
print("FINAL PRODUCTION FEATURES READY")
print("=" * 70)

print("Training shape:", X_final_train.shape)
print("Validation shape:", X_final_valid.shape)

FINAL PRODUCTION FEATURES READY
Training shape: (48000, 53)
Validation shape: (12000, 53)


In [32]:
# production model
print("=" * 70)
print("TRAINING FINAL PRODUCTION MODEL")
print("=" * 70)

y_final_train = np.log1p(
    final_train["posted_rate"].values
)

final_model = build_catboost(
    loss="RMSE",
    iterations=FINAL_MODEL_CONFIG["iterations"],
    learning_rate=FINAL_MODEL_CONFIG["learning_rate"],
    depth=FINAL_MODEL_CONFIG["depth"],
    l2_leaf_reg=FINAL_MODEL_CONFIG["l2_leaf_reg"],
    random_strength=FINAL_MODEL_CONFIG["random_strength"],
)

final_model.fit(
    X_final_train,
    y_final_train,
    cat_features=CATEGORICAL_FEATURES,
)

print("\nFinal production model trained successfully.")
print("Training rows:", len(final_train))
print("Iterations:", FINAL_MODEL_CONFIG["iterations"])

TRAINING FINAL PRODUCTION MODEL



Final production model trained successfully.
Training rows: 48000
Iterations: 178


In [33]:
# ===============================================================
# 20. FINAL VALIDATION PREDICTIONS
# ===============================================================

validation_log_pred = final_model.predict(
    X_final_valid
)

# Convert log-space predictions back to dollar-rate scale
validation_pred = np.expm1(
    validation_log_pred
)

# Safety checks
validation_pred = np.asarray(
    validation_pred,
    dtype=float
)

validation_pred = np.where(
    np.isfinite(validation_pred) & (validation_pred > 0),
    validation_pred,
    1e-6
)

print("=" * 70)
print("FINAL VALIDATION PREDICTIONS")
print("=" * 70)

print("Predictions:", len(validation_pred))
print("Min:", validation_pred.min())
print("Max:", validation_pred.max())
print("Mean:", validation_pred.mean())
print("Median:", np.median(validation_pred))

print("Finite:", np.isfinite(validation_pred).all())
print("Positive:", (validation_pred > 0).all())

FINAL VALIDATION PREDICTIONS
Predictions: 12000
Min: 211.2215040368152
Max: 6540.265879264999
Mean: 2347.851629881434
Median: 2035.3807723668356
Finite: True
Positive: True


In [37]:
# Load Spotter's original template
template_path = "data/validation-predictions-template.csv"

validation_template = pd.read_csv(template_path)

# Fill the blank predicted_rate column
validation_template["predicted_rate"] = validation_pred

# OVERWRITE the original Spotter template
validation_template.to_csv(
    template_path,
    index=False
)

print(" original template has been filled.")
display(validation_template.head(10))

 original template has been filled.


,load_id,predicted_rate
0,TE-000001,859.496282
1,TE-000002,4962.892284
2,TE-000003,5279.721108
3,TE-000004,4081.880165
4,TE-000005,1757.093975
5,TE-000006,4385.028363
6,TE-000007,5253.823849
7,TE-000008,2827.139887
8,TE-000009,1095.839082
9,TE-000010,1126.610310


In [35]:
print("=" * 70)
print("FINAL VALIDATION PREDICTION SANITY CHECK")
print("=" * 70)

print("\nPrediction quantiles:")
print(
    pd.Series(validation_pred).quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nTraining target quantiles:")
print(
    final_train["posted_rate"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nValidation input rows:", len(validation_df))
print("Prediction rows:", len(validation_pred))

print(
    "\nValidation prediction range:",
    validation_pred.min(),
    "to",
    validation_pred.max()
)
print(
    "Training target range:",
    final_train["posted_rate"].min(),
    "to",
    final_train["posted_rate"].max()
)

FINAL VALIDATION PREDICTION SANITY CHECK

Prediction quantiles:
0.01     337.643328
0.05     609.916856
0.25    1270.260401
0.50    2035.380772
0.75    3355.295680
0.90    4308.592865
0.95    4872.304635
0.99    5690.990747
dtype: float64

Training target quantiles:
0.01     327.1688
0.05     599.7385
0.25    1251.5550
0.50    2030.7600
0.75    3330.7500
0.90    4365.8230
0.95    4953.7665
0.99    5972.8340
Name: posted_rate, dtype: float64

Validation input rows: 12000
Prediction rows: 12000

Validation prediction range: 211.2215040368152 to 6540.265879264999
Training target range: 57.22 to 25533.0


In [43]:
december_df = pd.read_csv("data/december-chart-inputs.csv")

# Parse date
december_df["date"] = pd.to_datetime(
    december_df["date"],
    errors="coerce"
)

# Remove the blank output column from the supplied input
december_df = december_df.drop(
    columns=["predicted_rate"],
    errors="ignore"
)

# ------------------------------------------------------------
# 1. Add temporary IDs required by the existing feature code
# ------------------------------------------------------------
december_df["load_id"] = [
    f"DEC-{i:03d}"
    for i in range(1, len(december_df) + 1)
]

# ------------------------------------------------------------
# 2. Recover route coordinates from the labeled training data
#    Lexington -> Fort Wayne exists in the training data.
# ------------------------------------------------------------
geo_cols = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
]

route_geo = (
    final_train
    .groupby(["pickup", "delivery"])[geo_cols]
    .median()
    .reset_index()
)

december_df = december_df.merge(
    route_geo,
    on=["pickup", "delivery"],
    how="left"
)

# ------------------------------------------------------------
# 3. The supplied December file does not contain market_index
#    or quote_signal.
#
#    Use training-set medians as neutral fixed values.
#    This keeps the December scenario fixed except for DATE.
# ------------------------------------------------------------
december_df["market_index"] = final_train["market_index"].median()
december_df["quote_signal"] = final_train["quote_signal"].median()

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
required_columns = [
    "load_id",
    "pickup",
    "delivery",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "equipment",
    "weight",
    "date",
    "market_index",
    "quote_signal",
]

print("December shape:", december_df.shape)
print("Columns:", december_df.columns.tolist())

print("\nMissing values:")
print(december_df[required_columns].isna().sum())

display(december_df.head())

December shape: (31, 13)
Columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'load_id', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'market_index', 'quote_signal']

Missing values:
load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
dtype: int64


,pickup,delivery,distance,equipment,weight,date,load_id,pickup_lat,pickup_lon,delivery_lat,delivery_lon,market_index,quote_signal
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,DEC-001,36.99152,-84.99876,41.31561,-85.36206,1.0558,2.05575
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,DEC-002,36.99152,-84.99876,41.31561,-85.36206,1.0558,2.05575
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,DEC-003,36.99152,-84.99876,41.31561,-85.36206,1.0558,2.05575
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,DEC-004,36.99152,-84.99876,41.31561,-85.36206,1.0558,2.05575
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,DEC-005,36.99152,-84.99876,41.31561,-85.36206,1.0558,2.05575


In [44]:
X_final_train, X_december = attach_history_features(
    final_train,
    december_df,
    use_history=USE_HISTORY_FEATURES,
    high_threshold=8000.0,
)

print("Training features:", X_final_train.shape)
print("December features:", X_december.shape)

Training features: (48000, 53)
December features: (31, 53)


In [46]:
print("Same number of columns:",
      X_final_train.shape[1] == X_december.shape[1])

print("Same column names:",
      set(X_final_train.columns) == set(X_december.columns))

print("Same column order:",
      X_final_train.columns.tolist() == X_december.columns.tolist())

print("\nTraining categorical columns:")
print(CATEGORICAL_FEATURES)

print("\nTraining columns:")
print(X_final_train.columns.tolist())

print("\nDecember columns:")
print(X_december.columns.tolist())

Same number of columns: True
Same column names: True
Same column order: False

Training categorical columns:
['pickup', 'delivery', 'equipment', 'route']

Training columns:
['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'market_index', 'quote_signal', 'route', 'weight_raw', 'weight_missing', 'weight_negative', 'weight_clean', 'market_index_missing', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'days_since_start', 'distance_log1p', 'distance_sq', 'distance_x_market', 'distance_x_quote', 'market_x_quote', 'geodesic_distance_miles', 'distance_to_geodesic_ratio', 'route_hist_rpm_mean', 'route_hist_rate_mean', 'route_hist_high_rate_share', 'route_hist_count', 'pickup_hist_rpm_mean', 'pickup_hist_rate_mean', 'pickup_hist_high_rate_share', 'pickup_hist_count', 'delivery_hist_rpm_mean', 'delivery_hist_rate_mean', 'delivery_hist_hi

In [47]:
X_december = X_december.reindex(
    columns=X_final_train.columns
)

In [48]:
print(
    "Columns now identical:",
    X_final_train.columns.tolist() == X_december.columns.tolist()
)

Columns now identical: True


In [49]:
for col in CATEGORICAL_FEATURES:
    print(
        col,
        "| train dtype:", X_final_train[col].dtype,
        "| december dtype:", X_december[col].dtype
    )

pickup | train dtype: object | december dtype: object
delivery | train dtype: object | december dtype: object
equipment | train dtype: object | december dtype: object
route | train dtype: object | december dtype: object


In [ ]:
for col in CATEGORICAL_FEATURES:
    X_final_train[col] = X_final_train[col].astype(str)
    X_december[col] = X_december[col].astype(str)

In [51]:
december_log_pred = final_model.predict(X_december)

december_pred = np.expm1(december_log_pred)

december_pred = np.asarray(
    december_pred,
    dtype=float
)

december_pred = np.where(
    np.isfinite(december_pred) & (december_pred > 0),
    december_pred,
    1e-6
)

print("Number of predictions:", len(december_pred))
print("Minimum:", december_pred.min())
print("Maximum:", december_pred.max())
print("Mean:", december_pred.mean())

Number of predictions: 31
Minimum: 842.4860454249593
Maximum: 845.8637609109497
Mean: 844.1327719593619


In [52]:
route_check = final_train[
    (final_train["pickup"] == "Lexington") &
    (final_train["delivery"] == "Fort Wayne")
].copy()

print("Historical Lexington → Fort Wayne rows:", len(route_check))

if len(route_check) > 0:
    print("\nHistorical posted_rate:")
    print(route_check["posted_rate"].describe())

    print("\nHistorical distance:")
    print(route_check["distance"].describe())

    print("\nHistorical equipment:")
    print(route_check["equipment"].value_counts())

    print("\nHistorical weight:")
    print(route_check["weight"].describe())

    print("\nHistorical median rate:",
          route_check["posted_rate"].median())

    print("\nDecember prediction mean:",
          december_pred.mean())

Historical Lexington → Fort Wayne rows: 32

Historical posted_rate:
count     32.000000
mean     856.589062
std       63.194715
min      757.930000
25%      796.652500
50%      848.610000
75%      912.590000
max      973.750000
Name: posted_rate, dtype: float64

Historical distance:
count     32.000000
mean     363.668750
std        8.854612
min      346.300000
25%      358.150000
50%      362.300000
75%      368.150000
max      384.500000
Name: distance, dtype: float64

Historical equipment:
equipment
Dry Van    21
Reefer      8
Flatbed     3
Name: count, dtype: int64

Historical weight:
count       32.000000
mean     29185.625000
std       7279.155214
min      16195.000000
25%      23955.250000
50%      27336.500000
75%      36636.000000
max      41138.000000
Name: weight, dtype: float64

Historical median rate: 848.6099999999999

December prediction mean: 844.1327719593619


In [53]:
december_predictions = pd.read_csv("data/december-chart-inputs.csv")

december_predictions["predicted_rate"] = december_pred

december_predictions.to_csv(
    "december_predictions.csv",
    index=False
)

print(december_predictions)
print("\nSaved:", "december_predictions.csv")

       pickup    delivery  distance equipment  weight        date  \
0   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-01   
1   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-02   
2   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-03   
3   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-04   
4   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-05   
5   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-06   
6   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-07   
7   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-08   
8   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-09   
9   Lexington  Fort Wayne       360   Dry Van   32000  2025-12-10   
10  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-11   
11  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-12   
12  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-13   
13  Lexington  Fort Wayne       36